In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys

sys.path.insert(0, "../..")

In [2]:
import chex
import dill
import jax
import jax.numpy as jnp
import json
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scipy
import torch

from flax import nnx
from torch.utils.data import DataLoader
from typing import Any, NamedTuple
from typing_extensions import Protocol, runtime_checkable

from src.constants import *
from src.dataset import get_iter
from src.datasets.sum import Addition
from src.decoding import make_autoregressive
from src.utils import parse_dict

In [3]:
base_path = "/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/results"
algo_name = "addition-max_int_64-no_eos"
run_name = "ppo-scratch_1M-reset_immediately-0_cot_tokens-8x8-01-22-26_09_37_32-b4fba2ea-b795-46f4-a3ef-c9721684f647"

learner_path = os.path.join(base_path, algo_name, run_name)

eval_seed = 42
num_evals = 1
max_decode_len = 50
batch_size = 4
max_bits = 5

config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))
half_precision = config_dict["half_precision"]
dtype = jnp.bfloat16 if half_precision else jnp.float32

# Get dataset
max_int = int(2 ** max_bits)
config = parse_dict(config_dict)
dataset_kwargs = config.dataset_kwargs

## Rollout

In [4]:
if config.dataset_name == "curriculum":
    max_num_bits = int(np.ceil(np.log2(dataset_kwargs.datasets[-1]["dataset_kwargs"]["max_int"])))
    train_val_ratio = dataset_kwargs.datasets[-1]["dataset_kwargs"]["train_val_ratio"]
    predict_eos = dataset_kwargs.datasets[-1]["dataset_kwargs"]["predict_eos"]
    num_cot_tokens = dataset_kwargs.datasets[-1]["dataset_kwargs"]["num_cot_tokens"]
    right_to_left = dataset_kwargs.datasets[-1]["dataset_kwargs"]["right_to_left"]
    correctness_aware = dataset_kwargs.datasets[-1]["dataset_kwargs"]["correctness_aware"]
    carry_registers = dataset_kwargs.datasets[-1]["dataset_kwargs"]["carry_registers"]
else:
    max_num_bits = int(np.ceil(np.log2(dataset_kwargs.max_int)))
    train_val_ratio = dataset_kwargs.train_val_ratio
    predict_eos = dataset_kwargs.predict_eos
    num_cot_tokens = dataset_kwargs.num_cot_tokens
    right_to_left = dataset_kwargs.right_to_left
    correctness_aware = dataset_kwargs.correctness_aware
    carry_registers = getattr(dataset_kwargs, "carry_registers", False)

dataset = Addition(
    context_len=max_decode_len,
    max_int=max_int,
    train=False,
    seed=eval_seed,
    sequence_type="question_only",
    train_val_ratio=train_val_ratio,
    right_to_left=right_to_left,
    num_repeats=None,
    shuffle=False,
    exact=False,
    predict_eos=predict_eos,
    num_cot_tokens=num_cot_tokens,
    p_curriculum=0.0,
    p_inject_noop=0.0,
    max_noops=0,
    noop_as_pad=False,
    reverse_curriculum=False,
    correctness_aware=correctness_aware,
    carry_registers=carry_registers,
)

out_dim = int(dataset.output_space.n)
data_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
)
data_iter = get_iter(data_loader, None, dtype)
batch = next(data_iter)
batch = {
    k: np.repeat(v, num_evals, axis=0)
    for k, v in batch.items()
}

EOS TOKEN: 6
TOKEN MAP: {0: 0, 1: 1, 2: 2, 3: 3, 6: 6, 4: 4, 5: 5}


In [785]:
config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))

# Load model
last_step = sorted(os.listdir(os.path.join(learner_path, "models")))[-1]
train_state = dill.load(
    open(os.path.join(learner_path, "models", last_step), "rb")
)

model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)

rng = jax.random.PRNGKey(eval_seed)
rng, rollout_rng = jax.random.split(rng)

# Sample next batch and evaluate
_, init_cache = make_autoregressive(
    model,
    max_decode_len=max_decode_len,
    batch_size=batch_size * num_evals,
    embed_dim=config_dict["model_config"]["model_kwargs"]["embed_dim"],
    dtype=dtype,
    eval_mode=True,
)
cache = init_cache()
graphdef, _, rest = nnx.split(model, nnx.Cache, ...)

In [786]:
class StepState(NamedTuple):
    graphdef: Any
    rest: Any
    cache: Any
    eos_token: int
    rng: chex.PRNGKey
    observations: chex.Array
    actions: chex.Array
    solution: chex.Array
    solution_len: chex.Array
    solution_found: chex.Array
    pointer_correct: chex.Array
    last_prompt_idx: int
    eos: chex.Array
    step_i: int = 0
    deterministic: int = 0
    """
    XXX: Set both to zero if we're using CoT tokens or predicting <EOS>

    XXX: Assume that all possible responses contain tokens with ID up to max_token_id_to_shift
         - You will have to include max_token_id_to_shift + 1 CoT tokens to get the code to run
    """
    correct_aware_shift: int = 0
    max_token_id_to_shift: int = 0


class RolloutResult(NamedTuple):
    observations: chex.Array
    actions: chex.Array
    solution_found: chex.Array
    success: chex.Array
    response_length: chex.Array
    pointer_correct: chex.Array
    pred_mask: chex.Array
    last_prompt_idx: int
    eos: chex.Array


def predict_step(
    step_state: StepState,
):
    step_i = step_state.step_i
    rng, rng_step = jax.random.split(step_state.rng, 2)
    is_prompt = step_i < step_state.last_prompt_idx
    pointer_correct = step_state.pointer_correct
    solution_found = step_state.solution_found[:, step_i]

    # Autoregressively decode
    module = nnx.merge(step_state.graphdef, step_state.rest, step_state.cache)
    module.eval()
    module.set_attributes(deterministic=True, decode=True)
    logits = module({"sequence": step_state.observations[:, [step_i]],},)
    cache = nnx.state(module, nnx.Cache)

    logits = logits[:, 0]

    # Actual action taken, but output_tokens can be different
    action = jax.lax.cond(
        step_state.deterministic,
        lambda rng_step, logits: jnp.argmax(logits, axis=-1),
        jax.random.categorical,
        rng_step,
        logits,
    )

    # Shift token by some amount if we know the steps are wrong
    curr_soln_token = step_state.solution[
        jnp.arange(len(pointer_correct)), pointer_correct
    ]

    # Reset trajectory immediately
    not_prompt_pointer = jax.lax.select(
        curr_soln_token == action,
        pointer_correct + 1,
        jnp.ones_like(pointer_correct, dtype=int),
    )
    pointer_correct = jax.lax.select(
        is_prompt > 0.0,
        pointer_correct,
        not_prompt_pointer,
    )
    output_tokens = jnp.where(
        curr_soln_token != action,
        step_state.solution[:, 0],
        action,
    )

    solution_found = jnp.logical_or(
        solution_found,
        pointer_correct == step_state.solution_len
    )

    # Check for prompt boundary
    output_tokens = jnp.where(
        is_prompt,
        step_state.observations[:, step_i + 1],
        output_tokens,
    )

    # Check if the first EOS has been generated
    eos = jnp.logical_or(
        output_tokens == step_state.eos_token,
        step_state.eos[:, step_i],
    )

    # Update the entries on the i'th step
    observations = step_state.observations.at[:, step_i + 1].set(output_tokens)
    actions = step_state.actions.at[:, step_i + 1].set(action)
    solution_found = step_state.solution_found.at[:, step_i + 1].set(solution_found)
    eos = step_state.eos.at[:, step_i + 1].set(eos)

    step_state = StepState(
        graphdef=step_state.graphdef,
        rest=step_state.rest,
        cache=cache,
        eos_token=step_state.eos_token,
        rng=rng,
        observations=observations,
        actions=actions,
        solution=step_state.solution,
        solution_len=step_state.solution_len,
        solution_found=solution_found,
        pointer_correct=pointer_correct,
        last_prompt_idx=step_state.last_prompt_idx,
        eos=eos,
        step_i=step_i + 1,
        deterministic=step_state.deterministic,
        correct_aware_shift=step_state.correct_aware_shift,
        max_token_id_to_shift=step_state.max_token_id_to_shift,
    )

    return step_state

@nnx.jit
def rollout(
    graphdef: Any,
    cache: Any,
    rest: Any,
    rng: chex.PRNGKey,
    batch: Any,
    eos_token: int,
    deterministic: int = 0,
    correct_aware_shift: int = 0,
    max_token_id_to_shift: int = 0,
):
    question = batch["sequence"]
    solution = batch["target"]
    pointer_correct = batch["pointer_correct"]
    question_len = batch["question_len"]
    solution_len = batch["solution_len"]
    last_prompt_idx = question_len + pointer_correct - 1
    num_questions, max_step = question.shape

    step_state = StepState(
        graphdef=graphdef,
        rest=rest,
        cache=cache,
        eos_token=eos_token,
        rng=rng,
        observations=question,
        actions=jnp.zeros_like(question, dtype=int),
        solution=solution,
        solution_len=solution_len,
        solution_found=jnp.zeros_like(question, dtype=bool),
        pointer_correct=pointer_correct,
        last_prompt_idx=last_prompt_idx,
        eos=jnp.zeros((num_questions, max_step), dtype=bool),
        correct_aware_shift=correct_aware_shift,
        max_token_id_to_shift=max_token_id_to_shift,
        deterministic=deterministic,
    )

    step_state = jax.lax.while_loop(
        lambda state: jnp.logical_and(
            jnp.logical_not(jnp.all(state.solution_found[:, state.step_i])),
            state.step_i < max_step - 1,
        ),
        predict_step,
        step_state,
    )

    success = jnp.max(step_state.solution_found, axis=-1)
    solution_found = jnp.cumsum(step_state.solution_found, axis=-1) > 0
    solution_found_mask = success == 1
    response_length = (
        solution_found_mask * (jnp.argmax(solution_found, axis=-1) - question_len)
        + (1 - solution_found_mask) * (max_step - question_len - 1)
    )

    pred_mask = batch["mask"] - solution_found
    pred_mask = pred_mask == 1

    return RolloutResult(
        observations=step_state.observations[:, :-1],
        actions=step_state.actions[:, 1:],
        solution_found=solution_found[:, 1:],
        success=success,
        response_length=response_length,
        pred_mask=pred_mask[:, :-1],
        pointer_correct=step_state.pointer_correct,
        last_prompt_idx=step_state.last_prompt_idx,
        eos=step_state.eos[:, 1:],
    )

In [787]:
rollout_res = rollout(
    graphdef,
    cache,
    rest,
    rollout_rng,
    batch,
    eos_token=dataset.eos_token_id,
    deterministic=0,
    correct_aware_shift=dataset.correctness_aware_tokens_offset,
    max_token_id_to_shift=dataset.max_token_id_to_shift,
)

In [788]:
idx = 0

In [157]:
question_len = batch["question_len"][idx]

In [158]:
batch["target"][idx]

array([3, 1, 0, 1, 0, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
       6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
       6, 6, 6, 6, 6, 6, 6])

In [159]:
batch["mask"][idx], rollout_res.response_length[idx]

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       dtype=float32),
 Array(19, dtype=int32))

In [162]:
rollout_res.pred_mask[idx, question_len: question_len + rollout_res.response_length[idx]]

Array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True], dtype=bool)

In [163]:
rollout_res.actions[idx, question_len: question_len + rollout_res.response_length[idx]]

Array([0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1], dtype=int32)

In [164]:
len(rollout_res.observations[idx]) - (len(rollout_res.observations[idx, question_len: question_len + rollout_res.response_length[idx]]) + question_len)

np.int64(22)

In [121]:
rollout_res.observations[idx, question_len: question_len + rollout_res.response_length[idx]]

Array([3, 3, 3, 3, 3, 1, 3, 1, 0, 1, 3, 1, 0, 1, 3, 1, 0, 1, 0], dtype=int32)

In [131]:
rollout_res.solution_found[idx][question_len: question_len + rollout_res.response_length[idx]]

Array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True], dtype=bool)

In [134]:
rollout_res.pred_mask[idx][question_len: question_len + rollout_res.response_length[idx]]

Array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True], dtype=bool)

In [14]:
assert 0

AssertionError: 

## Compute returns

In [ ]:
def make_compute_returns(config, eos_token_id, reset_token_id, token_map):
    # Reward shaping
    reward_type = getattr(config, "reward_type", "default")
    if reward_type == "negative_on_failure":
        def shape_reward(batch, rollout_res):
            reward = jnp.zeros_like(rollout_res.actions)
            reward = reward.at[
                jnp.arange(len(rollout_res.response_length)),
                batch["question_len"] + rollout_res.response_length - 1
            ].set((-1) ** (1 - rollout_res.success))
            return reward

        def bandit_reward(rollout_res):
            return (-1) ** (1 - rollout_res.success)
    elif reward_type == "negative_dense":
        def shape_reward(batch, rollout_res):
            reward = jnp.full_like(rollout_res.actions, fill_value=-1)
            reward = reward.at[
                jnp.arange(len(rollout_res.response_length)),
                batch["question_len"] + rollout_res.response_length - 1
            ].set(rollout_res.success - 1)
            return reward

        def bandit_reward(rollout_res):
            return rollout_res.success - 1
    else:
        def shape_reward(batch, rollout_res):
            reward = jnp.zeros_like(rollout_res.actions)
            reward = reward.at[
                jnp.arange(len(rollout_res.response_length)),
                batch["question_len"] + rollout_res.response_length - 1
            ].set(rollout_res.success)
            return reward

        def bandit_reward(rollout_res):
            return rollout_res.success

    # Dr. GRPO
    dr_grpo = getattr(config, "dr_grpo", False)
    batch_size = getattr(config, "batch_size", 1)
    num_rollouts_per_sample = getattr(config, "num_rollouts_per_sample", 1)
    if dr_grpo and num_rollouts_per_sample > 1:
        def normalize_reward(rewards):
            group_changes = np.arange(0, batch_size, num_rollouts_per_sample)
            group_means = np.add.reduceat(rewards, group_changes) / num_rollouts_per_sample
            group_means = np.repeat(group_means, num_rollouts_per_sample, axis=0)
            rewards = rewards - group_means
            return rewards
    else:
        def normalize_reward(rewards):
            return rewards

    # MDP vs Bandit formulation
    if config.train_loss_config.mdp_type.startswith("episodic"):
        def _scan_monte_carlo_returns(
            rews: chex.Array,
            dones: chex.Array,
            gamma: float,
        ):
            def _returns(
                next_val, transition
            ):
                rew, done = transition
                val = (next_val * gamma) * (1 - done) + rew
                return val, val

            return jax.lax.scan(
                _returns,
                0,
                jnp.concatenate((rews, dones), axis=-1),
                len(rews),
                reverse=True,
            )[1]

        scan_monte_carlo_returns = jax.vmap(
            jax.jit(_scan_monte_carlo_returns),
            in_axes=[0, 0, None],
        )

        def process_reward(batch, rollout_res, reward, has_eos):
            returns = scan_monte_carlo_returns(
                reward[..., None],
                rollout_res.solution_found[..., None],
                config.gamma,
            )
            return returns
    elif config.train_loss_config.mdp_type.startswith("meta_rl"):
        # TODO: Use in-hindsight reward fraction
        def _scan_monte_carlo_meta_returns(
            rews: chex.Array,
            dones: chex.Array,
            resets: chex.Array,
            in_ep_gamma: float,
            cross_ep_gamma: float,
        ):
            def _returns(
                next_val, transition
            ):
                rew, done, reset = transition
                val = (
                    (1 - reset) * next_val * in_ep_gamma
                    + reset * next_val * cross_ep_gamma
                ) * (1 - done) + rew
                return val, val

            return jax.lax.scan(
                _returns,
                0,
                jnp.concatenate((rews, dones, resets), axis=-1),
                len(rews),
                reverse=True,
            )[1]

        scan_monte_carlo_meta_returns = jax.vmap(
            jax.jit(_scan_monte_carlo_meta_returns),
            in_axes=[0, 0, 0, None, None],
        )

        def process_reward(batch, rollout_res, reward, has_eos):
            returns = scan_monte_carlo_meta_returns(
                reward[..., None],
                rollout_res.solution_found[..., None],
                jnp.concatenate((
                    (rollout_res.observations == reset_token_id)[:, 1:],
                    jnp.full((len(reward), 1), fill_value=-1, dtype=int),
                ), axis=-1)[..., None],
                config.in_ep_gamma,
                config.cross_ep_gamma,
            )
            return returns
    elif config.train_loss_config.mdp_type.startswith("traj_improvement"):
        @jax.jit
        def scan_fn(carry, idx):
            pointer_correct = carry["pointer_correct"]
            actions = carry["actions"]
            target = carry["target"]
            pred_mask = carry["pred_mask"]
            last_reset_idx = carry["last_reset_idx"]
            reset_idxes = carry["reset_idxes"]
            correct_lens = carry["correct_lens"]
            curr_trial = carry["curr_trial"]

            action_match = actions[idx] == target[pointer_correct]
            is_reset = actions[idx] == reset_token_id
            is_reset_with_pred = jnp.logical_and(is_reset, pred_mask[idx])

            # Shift the pointer if the action matches the target and we're within a prediction mask
            reset_pointer = jax.lax.select(
                is_reset,
                1,
                0,
            )
            pointer_correct = jax.lax.select(
                action_match,
                pointer_correct + 1, # Increment pointer by 1 if current token matches
                reset_pointer, # Reset to 0 if it's a mistake and not a reset token, to 1 otherwise
            )

            # Update the last reset index to current index upon new trial
            last_reset_idx = jax.lax.select(
                is_reset_with_pred,
                idx,
                last_reset_idx,
            )

            reset_idxes = reset_idxes.at[curr_trial + 1].set(
                jax.lax.select(
                    is_reset_with_pred,
                    last_reset_idx,
                    reset_idxes[curr_trial + 1],
                )
            )

            correct_lens = correct_lens.at[curr_trial].set(
                jnp.maximum(pointer_correct, correct_lens[curr_trial])
            )

            curr_trial = jax.lax.select(
                is_reset_with_pred,
                curr_trial + 1,
                curr_trial,
            )

            return {
                "pointer_correct": pointer_correct,
                "last_reset_idx": last_reset_idx,
                "actions": actions,
                "target": target,
                "pred_mask": pred_mask,
                "reset_idxes": reset_idxes,
                "correct_lens": correct_lens,
                "curr_trial": curr_trial,
            }, None

        def process_reward(batch, rewards, response_lengths, has_eos):
            returns = np.zeros(batch["observations"].shape)
            for sample_i, (pred_mask, actions, target) in enumerate(zip(
                batch["pred_mask"], batch["actions"], batch["target"]
            )):
                pointer_correct = np.array(1, dtype=int)
                last_reset_idx = np.array(-1, dtype=int)
                curr_trial = np.array(0, dtype=int)
                reset_idxes = np.full_like(actions, fill_value=-1, dtype=int)
                reset_idxes[0] = np.where(pred_mask == 1)[0][0] - 1
                correct_lens = np.full_like(actions, fill_value=-1, dtype=int)
                last_idx = min(np.where(pred_mask == 1)[0][-1] + 2, actions.shape[-1])

                res, _ = jax.lax.scan(
                    scan_fn,
                    {
                        "pointer_correct": pointer_correct,
                        "last_reset_idx": last_reset_idx,
                        "actions": actions.at[:reset_idxes[0] + 1].set(reset_token_id),
                        "target": target,
                        "pred_mask": pred_mask.astype(int),
                        "reset_idxes": reset_idxes,
                        "correct_lens": correct_lens,
                        "curr_trial": curr_trial,
                    },
                    np.arange(last_idx),
                )

                reset_idxes = res["reset_idxes"]
                correct_lens = res["correct_lens"]
                correct_lens = np.concatenate(([0], correct_lens))

                # TODO: Negative reward
                answer_len = np.where(target == eos_token_id)[0]
                if len(answer_len) > 0:
                    answer_len = answer_len[0]
                else:
                    answer_len = len(target)

                last_idx = min(np.where(pred_mask == 1)[0][-1] + 1, actions.shape[-1])
                if getattr(config, "cumulative", True):
                    cum_correct_lens = np.maximum.accumulate(correct_lens)
                    improvements = (correct_lens[1:] - cum_correct_lens[:-1] - 1) / answer_len
                else:
                    improvements = correct_lens[1:] - correct_lens[:-1]
                reset_idxes = reset_idxes.at[(np.where(reset_idxes == -1))[0][0]].set(last_idx)
                trial_lengths = np.diff(reset_idxes[reset_idxes != -1])

                # Update the returns array
                num_trials = int(np.sum(reset_idxes != -1)) - 1
                if getattr(config, "discounting", True):
                    returns[sample_i, reset_idxes[0]:last_idx] = np.repeat(
                        config.gamma ** (
                            np.arange(num_trials)
                        ) * improvements[:num_trials],
                        trial_lengths,
                    )
                else:
                    # Regret like
                    returns[sample_i, reset_idxes[0]:last_idx] = np.repeat(
                        improvements[:num_trials] / (
                            np.arange(num_trials) + 1
                        ),
                        trial_lengths,
                    )
            return returns
    elif config.train_loss_config.mdp_type == "bandit":
        def process_reward(batch, rollout_res, reward, has_eos):
            returns = config.gamma ** (rollout_res.response_length - 1) * (bandit_reward(rollout_res) - (1 - has_eos))
            return returns
    else:
        raise NotImplementedError
    

    def compute_returns(batch, rollout_res):
        """
        Compute verifiable rewards
        Assume each token is an action, the state is the sequence up to this point
        The reward is based on whether there is a regex match with the target

        TODO: Entropy regularization objective
        """
        has_eos = jnp.max(jnp.logical_or(
            rollout_res.solution_found,
            rollout_res.eos,
        ), axis=-1)
        
        if not config.dataset_kwargs.predict_eos:
            has_eos = jnp.ones_like(has_eos, dtype=bool)

        reward = shape_reward(batch, rollout_res)
        reward = normalize_reward(reward)
        returns = process_reward(batch, rollout_res, reward, has_eos)

        return returns
    return compute_returns


In [832]:
config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))

config_dict["reward_type"] = "default"
# config_dict["reward_type"] = "negative_dense"
# config_dict["reward_type"] = "negative_on_failure"
# config_dict["train_loss_config"]["mdp_type"] = "episodic:length_bias_fix"
config_dict["train_loss_config"]["mdp_type"] = "meta_rl:length_bias_fix"
config_dict["in_ep_gamma"] = 1.0
config_dict["cross_ep_gamma"] = 0.9

# config_dict["train_loss_config"]["mdp_type"] = "bandit"
# config_dict["gamma"] = 0.99

config = parse_dict(config_dict)

compute_returns = make_compute_returns(
    config,
    dataset.eos_token_id,
    dataset.reset_token_id,
    dataset.token_map,
)

In [833]:
rollout_res.success

Array([ True,  True,  True,  True], dtype=bool)

In [834]:
rets = compute_returns(batch, rollout_res)

In [835]:
rets

Array([[0.43046713, 0.43046713, 0.43046713, 0.43046713, 0.43046713,
        0.43046713, 0.43046713, 0.43046713, 0.43046713, 0.47829682,
        0.5314409 , 0.5904899 , 0.6560999 , 0.7289999 , 0.7289999 ,
        0.80999994, 0.80999994, 0.80999994, 0.80999994, 0.9       ,
        0.9       , 0.9       , 0.9       , 1.        , 1.        ,
        1.        , 1.        , 1.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.5904899 , 0.5904899 , 0.5904899 , 0.5904899 , 0.5904899 ,
        0.5904899 , 0.5904899 , 0.5904899 , 0.5904899 , 0.6560999 ,
        0.7289999 , 0.80999994, 0.9       , 1.        , 1.        ,
        1.        , 1.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0. 

In [812]:
rollout_res.observations

Array([[0, 0, 0, 1, 2, 1, 0, 1, 1, 3, 3, 3, 3, 3, 1, 3, 1, 0, 1, 3, 1, 0,
        1, 3, 1, 0, 1, 0, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6],
       [0, 1, 1, 0, 2, 1, 0, 0, 1, 3, 3, 3, 3, 3, 1, 1, 1, 1, 3, 1, 3, 1,
        3, 1, 1, 3, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6],
       [1, 0, 1, 2, 1, 0, 0, 3, 0, 1, 1, 3, 0, 1, 1, 3, 0, 1, 1, 3, 0, 1,
        1, 3, 0, 1, 1, 3, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6],
       [1, 0, 0, 0, 1, 2, 1, 1, 0, 1, 0, 3, 0, 3, 0, 3, 0, 3, 0, 3, 0, 3,
        0, 3, 0, 0, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6]], dtype=int32)

In [584]:
idx = 0

In [585]:
rollout_res.success[idx]

Array(False, dtype=bool)

In [586]:
rollout_res.observations[idx, batch["question_len"][idx]:batch["question_len"][idx] + rollout_res.response_length[idx]]

Array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],      dtype=int32)

In [587]:
rollout_res.actions[idx, batch["question_len"][idx]:batch["question_len"][idx] + rollout_res.response_length[idx]]

Array([2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 0, 3, 2, 2, 3, 2, 3, 2, 3, 3, 2, 2,
       2, 2, 2, 3, 3, 3, 1, 2, 2, 2, 3, 3, 3, 2, 2, 2, 2, 3, 2],      dtype=int32)

In [588]:
batch["question_len"][idx], batch["sequence"][idx]

(np.int64(9),
 array([0, 0, 0, 1, 2, 1, 0, 1, 1, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6, 6]))

In [589]:
batch["target"][idx], batch["solution_len"][idx]

(array([3, 1, 0, 1, 0, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6, 6]),
 np.int64(6))

In [590]:
rollout_res.response_length[idx]

Array(41, dtype=int32)

In [591]:
rollout_res.pred_mask[idx]

Array([False, False, False, False, False, False, False, False, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True], dtype=bool)

In [592]:
returns[idx]

Array([-0. , -0. , -0. , -0. , -0. , -0. , -0. , -0. , -0. , -1. , -1. ,
       -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. ,
       -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. , -1. ,
       -1. , -1. , -1. , -1. , -0.9, -1. , -1. , -1. , -1. , -1. , -1. ,
       -1. , -1. , -1. , -1. , -1. , -1. ], dtype=float32)

In [583]:
power[idx]

Array([10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        1,  0,  0,  0,  0,  0,  1,  0,  2,  1,  0,  0,  1,  0,  1,  0],      dtype=int32)